# Bài 42: Phân đoạn ngữ nghĩa (Semantic Segmentation)

Phân loại gán nhãn cho cả một ảnh. Phát hiện (Bài 40-41) gán nhãn cho một nhúm hộp. **Phân đoạn ngữ nghĩa (semantic segmentation)** tiến thêm một bước: gán nhãn *mọi điểm ảnh* với một lớp/danh mục. Phương pháp học sâu đầu tiên là **mạng tích chập hoàn toàn (fully convolutional network — FCN)** (<a href="../references.html#long-2015-fcn">Long, Shelhamer & Darrell, 2015</a><span class="landmark-paper">&#9733;</span>), thay các lớp kết nối đầy đủ của một bộ phân loại bằng các phép tích chập, rồi tăng mẫu kết quả trở lại độ phân giải đầu vào. Đầu ra là một dự đoán dày đặc, theo từng điểm ảnh thay vì một nhãn duy nhất cho cả ảnh. Bài này xây dựng ý tưởng đó từ đầu, đối chiếu với một FCN tiền huấn luyện thật, rồi thêm vào một cải tiến cụ thể — các kết nối tắt.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['bicycle_train.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Một nhiệm vụ gán nhãn điểm ảnh 3 lớp

Tạo một ảnh với vài hình tròn và hình vuông nhỏ rải rác trên nền. Mục tiêu là một bản đồ lớp đầy đủ theo từng điểm ảnh: 0 = nền, 1 = hình tròn, 2 = hình vuông.

In [ ]:
def make_scene(rng, size=32, n_shapes=5):
    scene = np.zeros((size, size), dtype=np.float32)
    mask = np.zeros((size, size), dtype=np.int64)  # 0=background, 1=circle, 2=square
    # Circles and squares are drawn at different fixed intensities (0.85 vs.
    # 1.0) rather than both at 1.0. Where shapes overlap, whichever is drawn
    # last wins in BOTH scene and mask -- so the pixel's intensity now always
    # matches its label, even in the overlap, instead of being ambiguous.
    intensity = {1: 0.85, 2: 1.0}
    for _ in range(n_shapes):
        shape_type = rng.choice([1, 2])
        r = rng.integers(2, 4)
        cx, cy = rng.integers(r, size - r), rng.integers(r, size - r)
        yy, xx = np.mgrid[0:size, 0:size]
        if shape_type == 1:
            m = ((xx - cx) ** 2 + (yy - cy) ** 2) <= r ** 2
        else:
            m = (np.abs(xx - cx) <= r) & (np.abs(yy - cy) <= r)
        scene[m] = intensity[shape_type]
        mask[m] = shape_type
    scene = np.clip(scene + rng.normal(0, 0.05, scene.shape), 0, 1).astype(np.float32)
    return scene, mask

rng = np.random.default_rng(13)
N = 300
scenes, masks = [], []
for _ in range(N):
    s, m = make_scene(rng)
    scenes.append(s); masks.append(m)
scenes = np.array(scenes, dtype=np.float32)
masks = np.array(masks, dtype=np.int64)

split = int(0.85 * N)
Xtr, Mtr = scenes[:split], masks[:split]
Xte, Mte = scenes[split:], masks[split:]

class_fracs = (np.bincount(Mtr.ravel()) / Mtr.size).round(3).tolist()
print('class pixel fractions (train):', dict(zip(['background', 'circle', 'square'], class_fracs)))

fig, axes = plt.subplots(2, 4, figsize=(9, 4.5))
for i in range(4):
    axes[0, i].imshow(Xtr[i], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(Mtr[i], cmap='viridis', vmin=0, vmax=2); axes[1, i].axis('off')
axes[0, 0].set_title('input', fontsize=9, loc='left')
axes[1, 0].set_title('per-pixel ground truth label', fontsize=9, loc='left')
plt.show()

## Một mạng kiểu FCN tối giản

Một mạng phân đoạn đưa ra một vector xác suất lớp cho mọi điểm ảnh, nên đầu ra của nó có cùng kích thước không gian với đầu vào. **Bộ mã hóa (encoder)** là một CNN thông thường, giảm mẫu hai lần qua gộp cực đại (Bài 34) để xây dựng các đặc trưng giàu ngữ nghĩa với trường tiếp nhận rộng (kim tự tháp ở Bài 11). **Bộ giải mã (decoder)** sau đó tăng mẫu phần thắt cổ chai (bottleneck) thô đó thẳng trở lại độ phân giải đầy đủ. Đó là toàn bộ công thức FCN: chỉ là giảm mẫu rồi tăng mẫu.

In [ ]:
class FCNTiny(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec2 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.Conv2d(32, 16, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(16, n_classes, 1)

    def forward(self, x):
        f1 = self.enc1(x)                # (B,16,H,W)
        f2 = self.enc2(self.pool(f1))    # (B,32,H/2,W/2)
        f3 = self.enc3(self.pool(f2))    # (B,64,H/4,W/4)  -- the bottleneck
        d2 = self.dec2(self.up(f3))      # upsample only -- no fusion with f2
        d1 = self.dec1(self.up(d2))      # upsample only -- no fusion with f1
        return self.out(d1)

def mean_iou(preds, targets, n_classes=3):
    ious = []
    for c in range(n_classes):
        p, t = preds == c, targets == c
        inter, union = (p & t).sum(), (p | t).sum()
        ious.append(inter / union if union > 0 else float('nan'))
    return ious

torch.manual_seed(0)
model_fcn = FCNTiny()
opt = torch.optim.Adam(model_fcn.parameters(), lr=0.01)
Xt = torch.tensor(Xtr).unsqueeze(1); Mt = torch.tensor(Mtr)
for _ in range(200):
    opt.zero_grad()
    loss = F.cross_entropy(model_fcn(Xt), Mt)
    loss.backward()
    opt.step()

with torch.no_grad():
    preds_fcn = model_fcn(torch.tensor(Xte).unsqueeze(1)).argmax(1).numpy()

pixel_acc_fcn = (preds_fcn == Mte).mean()
ious_fcn = mean_iou(preds_fcn, Mte)
print(f'pixel accuracy: {pixel_acc_fcn:.1%}')
for name, iou in zip(['background', 'circle', 'square'], ious_fcn):
    print(f'  {name:>10} IoU: {iou:.3f}')
print(f'mean IoU: {np.mean(ious_fcn):.3f}')

Ảnh động bên dưới theo dõi một ảnh đi qua mạng từng giai đoạn, cho thấy độ phân giải thu nhỏ dần qua bộ mã hóa (độ phân giải đầy đủ → một nửa → thắt cổ chai một phần tư) và được tăng mẫu trở lại qua bộ giải mã. Hai khung hình cuối cho thấy các xác suất softmax thô (một sự pha trộn liên tục của ba màu lớp ở bất cứ đâu mạng không chắc chắn), rồi dự đoán `argmax` cứng thực sự được dùng, ép mọi điểm ảnh về một màu thuần. Phép ép đó là lý do kết quả trông thô hơn các đặc trưng giải mã chi tiết mượt mà ngay trước nó — đó không phải mất độ phân giải, mà là một bản đồ liên tục bị ép thành một nhãn rời rạc.

<img src="../animations/fcn_encoder_decoder_schematic_demo.gif" alt="Ảnh động theo dõi một ảnh qua FCNTiny: cảnh đầu vào được xử lý từng giai đoạn, thu nhỏ từ độ phân giải đầy đủ 32x32 qua 16x16 xuống phần thắt cổ chai 8x8 trong bộ mã hóa, rồi tăng mẫu trở lại qua 16x16 lên 32x32 trong bộ giải mã, rồi hiển thị sự pha trộn xác suất lớp softmax liên tục, và cuối cùng là dự đoán lớp theo từng điểm ảnh argmax cứng, với một chỉ báo tiến trình cho thấy giai đoạn hiện tại trong quy trình" style="display: block; margin: 0 auto; max-width: 620px;">

## Trong thực tế: FCN trên một bức ảnh thật

`torchvision` cung cấp **FCN** (<a href="../references.html#long-2015-fcn">Long et al., 2015</a><span class="landmark-paper">&#9733;</span>), được tiền huấn luyện trên ảnh COCO (gán nhãn theo 21 danh mục của Pascal VOC &mdash; 20 lớp đối tượng cộng với nền). Về kiến trúc, đây là cùng công thức với `FCNTiny` ở trên: bộ mã hóa, rồi tăng mẫu.

In [ ]:
import torchvision
import cv2
import matplotlib.patches as mpatches

weights = torchvision.models.segmentation.FCN_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1
fcn = torchvision.models.segmentation.fcn_resnet50(weights=weights)
fcn.eval()  # frozen, no training at all
voc_classes = weights.meta['categories']

photo = cv2.imread('../img/bicycle_train.jpg')
photo_rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)

preprocess = weights.transforms()
photo_tensor = torch.tensor(photo_rgb / 255.0, dtype=torch.float32).permute(2, 0, 1)
batch = preprocess(photo_tensor).unsqueeze(0)

with torch.no_grad():
    real_out = fcn(batch)['out']
real_pred = real_out.argmax(1)[0].numpy()
# the model's internal resolution differs from the photo's; resize the label map back up to match
real_pred_full = cv2.resize(real_pred.astype(np.uint8), (photo_rgb.shape[1], photo_rgb.shape[0]),
                             interpolation=cv2.INTER_NEAREST)

found_ids = sorted(c for c in np.unique(real_pred_full) if c != 0)
colors = plt.cm.tab10(np.linspace(0, 1, 10))

label_rgb = np.full((*real_pred_full.shape, 3), 255, dtype=np.uint8)  # background = white
for i, cls_id in enumerate(found_ids):
    color_rgb = (np.array(colors[i][:3]) * 255).astype(np.uint8)
    label_rgb[real_pred_full == cls_id] = color_rgb

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(photo_rgb); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(label_rgb); axes[1].set_title('FCN per-pixel labels'); axes[1].axis('off')
legend_handles = [mpatches.Patch(color=colors[i], label=voc_classes[c]) for i, c in enumerate(found_ids)]
axes[1].legend(handles=legend_handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

<p class="image-attribution">Ảnh của <a href="https://unsplash.com/@fhanspach?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText" target="_blank" rel="noopener">Felix Hanspach</a> trên <a href="https://unsplash.com/photos/people-riding-on-yellow-tram-during-night-time-zoKDZXcjp1s?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText" target="_blank" rel="noopener">Unsplash</a></p>

Người đi xe đạp được gán nhãn gọn gàng là `person`, với các bánh xe đạp được gán nhãn đúng là `bicycle` ngay bên dưới — cả hai đều là các phát hiện thật, đặt đúng chỗ. Chiếc xe điện (tram) mới là chỗ thú vị: mô hình chia nó giữa `train` (phần trên, có cửa sổ) và `bus` (phần thân dưới) thay vì chọn một nhãn cho cả phương tiện. Đó thực ra không hẳn là lỗi mà là sự phản ánh trung thực vốn từ vựng hạn chế của VOC — một chiếc xe điện thực sự là trường hợp mơ hồ giữa "tàu hỏa" và "xe buýt" đối với một mô hình chưa bao giờ được cho một lớp "xe điện" để chọn, và hai phỏng đoán này gần như bám theo một đường nối thị giác thực sự trên phương tiện (cửa sổ và thân xe) chứ không phải ngẫu nhiên. Một đốm `person` thứ hai, nhỏ hơn ở cạnh phải bắt được người đi bộ bị mờ ở phía sau.

## Thêm kết nối tắt: U-Net

Bộ giải mã của `FCNTiny` chỉ thấy các đặc trưng thắt cổ chai đã gộp, đã tăng mẫu. Sau hai vòng gộp, phần thắt cổ chai đó chỉ còn một phần tư độ phân giải không gian của đầu vào, và các hình nhỏ hoặc xếp sát nhau có thể nhòe vào nhau hoặc mất hoàn toàn ranh giới ở độ phân giải đó. **U-Net** (<a href="../references.html#ronneberger-2015">Ronneberger, Fischer & Brox, 2015</a><span class="landmark-paper">&#9733;</span>) khắc phục điều này bằng **các kết nối tắt (skip connections)**: ở mỗi giai đoạn giải mã, nó nối bản đồ đặc trưng của bộ mã hóa ở *độ phân giải tương ứng*, bảo toàn các chi tiết mịn mà nếu không sẽ bị mất trong quá trình gộp. Điều này tương tự về tinh thần với các kết nối dư ở Bài 37, nhưng mục đích và cơ chế khác nhau: U-Net nối (concatenate) các đặc trưng qua ranh giới mã hóa-giải mã, thay vì cộng một cập nhật dư bên trong một chồng mạng duy nhất.

In [ ]:
class UNetTiny(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(32, 64, 3, padding=1), nn.ReLU())
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.dec2 = nn.Sequential(nn.Conv2d(64 + 32, 32, 3, padding=1), nn.ReLU())
        self.dec1 = nn.Sequential(nn.Conv2d(32 + 16, 16, 3, padding=1), nn.ReLU())
        self.out = nn.Conv2d(16, n_classes, 1)

    def forward(self, x):
        f1 = self.enc1(x)                                     # (B,16,H,W)
        f2 = self.enc2(self.pool(f1))                          # (B,32,H/2,W/2)
        f3 = self.enc3(self.pool(f2))                          # (B,64,H/4,W/4)
        d2 = self.dec2(torch.cat([self.up(f3), f2], dim=1))    # skip from f2
        d1 = self.dec1(torch.cat([self.up(d2), f1], dim=1))    # skip from f1
        return self.out(d1)

torch.manual_seed(0)
model_unet = UNetTiny()
opt2 = torch.optim.Adam(model_unet.parameters(), lr=0.01)
for _ in range(200):
    opt2.zero_grad()
    loss = F.cross_entropy(model_unet(Xt), Mt)
    loss.backward()
    opt2.step()

with torch.no_grad():
    preds_unet = model_unet(torch.tensor(Xte).unsqueeze(1)).argmax(1).numpy()

pixel_acc_unet = (preds_unet == Mte).mean()
ious_unet = mean_iou(preds_unet, Mte)

print(f'{"":>18} {"pixel acc":>10} {"mean IoU":>10}')
print(f'{"FCN (no skip)":>18} {pixel_acc_fcn:>10.1%} {np.mean(ious_fcn):>10.3f}')
print(f'{"U-Net (skip)":>18} {pixel_acc_unet:>10.1%} {np.mean(ious_unet):>10.3f}')

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(9, 9))
for i in range(4):
    axes[0, i].imshow(Xte[i], cmap='gray')
    axes[1, i].imshow(Mte[i], cmap='viridis', vmin=0, vmax=2)
    axes[2, i].imshow(preds_fcn[i], cmap='viridis', vmin=0, vmax=2)
    axes[3, i].imshow(preds_unet[i], cmap='viridis', vmin=0, vmax=2)
    for r in range(4):
        axes[r, i].axis('off')
for r, name in enumerate(['input', 'ground truth mask', 'pred (FCN)', 'pred (U-Net)']):
    axes[r, 0].set_title(name, fontsize=9, loc='left')
plt.tight_layout()
plt.show()

Ảnh động bên dưới cho thấy cả hai bộ giải mã được huấn luyện trên cùng bốn cảnh được giữ lại này: dự đoán thắt cổ chai của FCN đơn thuần vẫn thô và làm nhòe các hình xếp sát nhau, trong khi các kết nối tắt của U-Net cho phép nó khôi phục các ranh giới sắc nét khi huấn luyện tiến triển.

<img src="../animations/fcn_unet_training_demo.gif" alt="Ảnh động huấn luyện FCNTiny và UNetTiny cạnh nhau trên bốn cảnh kiểm tra được giữ lại: cả hai bắt đầu dự đoán toàn nền, rồi khi huấn luyện tiến triển dự đoán của FCN vẫn mờ và gộp các hình gần nhau lại, trong khi dự đoán của U-Net sắc nét dần thành gần như khớp chính xác với các mặt nạ chuẩn" style="display: block; margin: 0 auto; max-width: 820px;">

Với các hình nhỏ, xếp sát nhau, kết nối tắt mang lại một cải thiện IoU trung bình đáng kể so với một FCN đơn thuần. Một FCN không thể khôi phục chi tiết đã bị mất trong phần thắt cổ chai độ phân giải thấp, vì phép nội suy chỉ có thể làm việc trên thông tin sẵn có. Kết nối tắt khắc phục vấn đề này bằng cách cho bộ giải mã một đường đi thứ hai, không qua thắt cổ chai, tới các đặc trưng độ phân giải cao của bộ mã hóa, để các ranh giới chính xác không bao giờ phải sống sót qua phần thắt cổ chai ngay từ đầu.

Điều này giúp giải thích tại sao U-Net vẫn phổ biến cho các nhiệm vụ đòi hỏi ranh giới chính xác, đặc biệt là ảnh y tế. Nhưng kết nối tắt không phổ quát: các mạng phổ biến như **FCN** đơn thuần và **DeepLabV3** không có kết nối tắt nào, và ngay cả **DeepLabV3+** cũng chỉ thêm một kết nối duy nhất, không phải kết nối kiểu U-Net đầy đủ ở mọi độ phân giải. Vì vậy, dù kết nối tắt là một công cụ hữu ích, chúng không phải là yêu cầu bắt buộc &mdash; giá trị của chúng phụ thuộc vào mức độ chi tiết không gian mịn mà nhiệm vụ đòi hỏi.

### Bài tập

1. Tăng `n_shapes` từ 5 lên 10, làm cảnh đông đúc hơn. Khoảng cách IoU trung bình giữa `FCNTiny` và `UNetTiny` lớn hơn hay nhỏ hơn? Điều đó gợi ý gì về thời điểm kết nối tắt quan trọng nhất?
2. Hàm mất mát của bài này là entropy chéo theo từng điểm ảnh đơn thuần. In các trọng số lớp theo từng điểm ảnh ngụ ý bởi `class pixel fractions` ở trên và thử `F.cross_entropy(logits, Mt, weight=inverse_class_freq)` (cách khắc phục mất cân bằng ở Bài 36, áp dụng ở đây) để xem nó có thay đổi sự cân bằng IoU giữa hình tròn/hình vuông không.
3. `nn.Upsample(mode='nearest')` được dùng cho đơn giản. Thay vào đó hãy thử `mode='bilinear', align_corners=False` (nội suy song tuyến tính ở Bài 9, giờ nằm bên trong một mạng) và so sánh IoU trung bình cho cả `FCNTiny` và `UNetTiny`. Phép tăng mẫu mượt hơn giúp ích nhiều hơn hay ít hơn so với kết nối tắt?
4. In độ tin cậy (xác suất softmax của lớp được dự đoán) tại vài điểm ảnh dọc theo đường nối nơi kết quả FCN thật chuyển từ `train` sang `bus` trên chiếc xe điện. Mô hình chia tách một cách tự tin, hay thực sự không chắc chắn ngay tại ranh giới đó — và bạn kỳ vọng gì từ một mạng được yêu cầu chọn giữa hai lớp khi đối tượng thực không khớp chính xác với lớp nào?